In [2]:
# Cập nhật danh sách gói (bắt buộc)
!sudo apt-get update

# Cài đặt công cụ zstd để giải nén file .tar.zst của Ollama phiên bản mới
!sudo apt-get install -y zstd

# Cài đặt các thư viện Python
%pip install ollama

Get:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:2 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Get:3 https://cli.github.com/packages stable InRelease [3,917 B]
Get:4 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]      
Hit:5 http://archive.ubuntu.com/ubuntu jammy InRelease                         
Get:6 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]           
Get:7 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]        
Get:8 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [93.4 kB]
Get:9 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]      
Get:10 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Get:11 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [2,615 kB]
Get:12 https://cli.github.com/packages stable/main amd64 Packages [356 B]

In [3]:
# Chạy script cài đặt Ollama (lúc này đã có zstd để giải nén)
!curl -fsSL https://ollama.com/install.sh | sh

>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%                                        9.4%                           19.8%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


In [4]:
#chạy server ollama

import subprocess
import time

# (Tùy chọn) Kill tiến trình ollama cũ nếu có, để tránh xung đột cổng
subprocess.run("pkill ollama", shell=True, stderr=subprocess.DEVNULL)
time.sleep(1)

# Khởi động ollama serve trong background
# stdout và stderr được chuyển hướng để tránh làm lộn xộn output của notebook
subprocess.Popen("ollama serve", shell=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

# Đợi vài giây cho server khởi động hoàn toàn
time.sleep(4)
print("✅ Ollama server đã được khởi động thành công!")


✅ Ollama server đã được khởi động thành công!


In [5]:
#tải model

!ollama pull qwen2.5:3b
!ollama pull qwen2.5:7b

]11;?\pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠴ pulling manifest ⠦ pulling manifest 
pulling 5ee4f07cdb9b:   0% ▕                  ▏  58 KB/1.9 GB                  pulling manifest 
pulling 5ee4f07cdb9b:   0% ▕                  ▏ 3.9 MB/1.9 GB                  pulling manifest 
pulling 5ee4f07cdb9b:   3% ▕                  ▏  63 MB/1.9 GB                  pulling manifest 
pulling 5ee4f07cdb9b:   8% ▕█                 ▏ 147 MB/1.9 GB                  pulling manifest 
pulling 5ee4f07cdb9b:  10% ▕█                 ▏ 192 MB/1.9 GB                  pulling manifest 
pulling 5ee4f07cdb9b:  15% ▕██                ▏ 282 MB/1.9 GB                  pulling manifest 
pulling 5ee4f07cdb9b:  19% ▕███               ▏ 364 MB/1.9 GB                  pulling manifest 
pulling 5ee4f07cdb9b:  21% ▕███               ▏ 406 MB/1.9 GB                  pulling manifest 
pulling 5ee4f07cdb9b:  26% ▕████              ▏ 500 MB/1.9 GB    

In [4]:
#check

!ollama list

/bin/bash: line 1: ollama: command not found


In [7]:
from ollama import generate
from pydantic import BaseModel
from pathlib import Path

def load_prompt(filename: str,path: str, **kwargs) -> str:
    prompt_path = Path(path+ filename)
    template = prompt_path.read_text(encoding="utf-8")
    # Nếu có placeholder, thay thế bằng kwargs
    for key, value in kwargs.items():
        placeholder = "{{" + key + "}}"
        template = template.replace(placeholder, value)
    return template


def problem_extracter(question: str):
    problem_text = (
    question
    )
    prompt_text = load_prompt(filename="prompt.txt",path="/kaggle/input/datasets/nam1706/testing-v1-2/", problem=problem_text)
    prompt_text = question
    response = generate(
        model='qwen2.5:7b',
        prompt=prompt_text,
        options={
            'temperature': 0,
            'keep_alive':'5m',
            'num_predict': 4000
        },
    )
    raw = response
    return raw['response']

In [63]:
#test hàm extracting
problem_text = """You are a physics problem parser. Your goal is to convert a word problem into a structured JSON object describing the fixed charges and the target point where the field (or force) is to be evaluated. You may reason briefly before the final output, but your last line must be exactly "output:" followed by the JSON on the next lines. Do NOT include any other text after the JSON.

RULES:
1. Convert all units to SI: charge in Coulombs (C), distance in meters (m).
2. Choose a convenient Cartesian coordinate system. Place the origin and axes to simplify the description (e.g., align a segment with the x-axis).
3. All charges with known, fixed positions go into the "charges" array. Each entry has:
   - "id": a string (like "q1", "q2")
   - "value": number (charge in C)
   - "position": [x, y] (numbers), based on the origin of the coordinate system.
   - "point": which point is it infer from the problem(like "A","B",..). Put randomly if the problem doesnt state.
4. The target point (where the electric field is to be computed, or where a charge is placed to compute the force on it) is described in "target_point_spec". This object has:
   - "type": either "explicit" or "equations"
   - If "explicit": include "coordinates": [x, y]
   - If "equations": include "equations": ["eq1", "eq2", ...]  (use the variables x and y for the point's coordinates)
   - Optionally, if a specific charge is placed at the target point (i.e., the problem asks for the force ON that charge), add these fields below to "target_point_spec":
        "target_charge_id": string ("q3","q0",...)
        "target_charge_value": number
     Do NOT add that charge to the "charges" array – only describe it here.
   - "comment": a short explanation
5. How to decide between "explicit" and "equations":
   - Use "explicit" if the coordinates can be directly determined from simple geometric clues (e.g., "on the perpendicular bisector at 4 cm from the midpoint", "at the midpoint of AB", "vertex of an equilateral triangle with known side").
   - Use "equations" if the point is defined **only** by its distances to two or more known points, without any directional information (e.g., "AC = 12 cm, BC = 16 cm"). Even if the distances satisfy a Pythagorean triple and the coordinates *could* be calculated, you must NOT do it; instead, output the distance equations as algebraic constraints. This preserves the problem as a system to be solved later.
6. Do NOT include any field or force formulas, constants (k, ε₀), or numerical solutions.

---

EXAMPLES:

Example 1:
Problem: Two point charges, q1 = 0.5 nC and q2 = –0.5 nC, are placed at points A and B, separated by 6 cm in air. What is the magnitude of the electric field strength at point M, which is located on the perpendicular bisector of AB, at a distance ℓ = 4 cm from the midpoint of AB?
output:
{
  "charges": [
    {"id": "q1", "value": 5e-10, "position": [-0.03, 0], "point":"A"},
    {"id": "q2", "value": -5e-10, "position": [0.03, 0], "point":"B"}
  ],
  "target_point_spec": {
    "type": "explicit",
    "coordinates": [0, 0.04],
    "comment": "Midpoint of AB is origin, y-axis perpendicular bisector."
  },
  "comment": "Origin at midpoint of AB, x-axis along AB."
}

Example 2:
Problem: Three electric charges, q1 = q2 = q3 = 1.6 × 10^-19 C, are placed at the three vertices of an equilateral triangle ABC with side length 16 cm in air. Determine the net electric force vector acting on q3.
output:
{
  "charges": [
    {"id": "q1", "value": 1.6e-19, "position": [0, 0], "point":"A"},
    {"id": "q2", "value": 1.6e-19, "position": [0.16, 0], "point":"B"}
  ],
  "target_point_spec": {
    "type": "equations",
    "equations": [
      "(x - 0.16)^2 + y^2 = 0.16^2"
      "x^2 + y^2 = 0.16^2"
    ],
    "target_charge_id": "q3",
    "target_charge_value": 1.6e-19,
    "comment": "C=(x,y) is defined by distances to A(0,0)=0.16 m and B(0.16,0)=0.16 m."
  },
  "comment": "Origin at q1, x-axis along AB."
}

Example 3:
Problem: Two point charges, q1 = 4 × 10^-6 C and q2 = -6.4 × 10^-6 C, are placed at A and B separated by 20 cm in air. Determine the electric field strength at point C, knowing that AC = 12 cm and BC = 16 cm.
output:
{
  "charges": [
    {"id": "q1", "value": 4e-6, "position": [0, 0], "point":"A"},
    {"id": "q2", "value": -6.4e-6, "position": [0.2, 0], "point":"B"}
  ],
  "target_point_spec": {
    "type": "equations",
    "equations": [
      "x^2 + y^2 = 0.12^2",
      "(x - 0.2)^2 + y^2 = 0.16^2"
    ],
    "comment": "C=(x,y) is defined by distances to A(0,0)=0.12 m and B(0.2,0)=0.16 m."
  },
  "comment": "Origin at A, x-axis along AB."
}

Example 4:
Problem: Two charges, q1 = 6 × 10^-8 C and q2 = -6 × 10^-8 C, are placed at points A and B in air, 8 cm apart. A third charge, q = 6 × 10^-8 C, is placed at point C, with CA = 5 cm and CB = 3 cm. Determine the force acting on q3.
output:
{
  "charges": [
    {"id": "q1", "value": 6e-8, "position": [0, 0], "point":"A"},
    {"id": "q2", "value": -6e-8, "position": [0.08, 0], "point":"B"}
  ],
  "target_point_spec": {
    "type": "equations",
    "equations": [
      "x^2 + y^2 = 0.05^2",
      "(x - 0.08)^2 + y^2 = 0.03^2"
    ],
    "target_charge_id": "q",
    "target_charge_value": 6e-8,
    "comment": "C is defined by distances CA=0.05 m, CB=0.03 m. q3 sits at C."
  },
  "comment": "Origin at A, x-axis along AB."
}

Example 5:
Problem: At two points A and B, separated by 10 cm in air, two electric charges q1 = q2 = 16 × 10^-8 C are placed. Given point C that AC = BC = 8 cm. Determine the electric field strength at C. (No charge at C.)
output:
{
  "charges": [
    {"id": "q1", "value": 1.6e-7, "position": [-0.05, 0], "point":"A"},
    {"id": "q2", "value": 1.6e-7, "position": [0.05, 0], "point":"B"}
  ],
  "target_point_spec": {
    "type": "equations",
    "equations": [
      "(x + 0.05)^2 + y^2 = 0.08^2",
      "(x - 0.05)^2 + y^2 = 0.08^2"
    ],
    "comment": "C is equidistant from A(-0.05,0) and B(0.05,0) at 0.08 m."
  },
  "comment": "Origin at midpoint of AB (AB=0.1 m), x-axis along AB."
}

---

Now, process the following problem. You can reasoning, then provide exactly the line "output:" followed by the JSON.

Problem: {{	Three electric charges, q1 = q2 = q3 = 1.6 × 10^-19 C, are placed at the three vertices of an equilateral triangle ABC with side length 16 cm in air. Determine the net electric force vector acting on q3.}}"""
problem_extracter(problem_text)

'output:\n{\n  "charges": [\n    {"id": "q1", "value": 1.6e-19, "position": [0, 0], "point":"A"},\n    {"id": "q2", "value": 1.6e-19, "position": [0.16, 0], "point":"B"}\n  ],\n  "target_point_spec": {\n    "type": "equations",\n    "equations": [\n      "(x - 0.16 * cos(120)) ^ 2 + y^2 = (0.16 / 2) ^ 2",\n      "x^2 + y^2 = (0.16 / 2) ^ 2"\n    ],\n    "target_charge_id": "q3",\n    "target_charge_value": 1.6e-19,\n    "comment": "C=(x,y) is defined by distances to A(0,0)=0.08 m and B(0.16,0)=0.08 m, forming an equilateral triangle."\n  },\n  "comment": "Origin at q1, x-axis along AB."\n}'

In [32]:
# tra duong dan, ae doi ten "nam1706" thanh ten kaggle roi copy mang xuong cell duoi
import os
print(os.listdir('/kaggle/input/datasets/nam1706/testing-v1-1'))


['prompt.txt']


In [62]:
import json
import re
import numpy as np
import sympy as sp


# Hằng số Coulomb
k = 9e9

def validate_parsed_json(data):
    """
    Kiểm tra cấu trúc và kiểu dữ liệu của JSON đầu ra.
    Nếu không hợp lệ, ném ValueError.
    """
    if not isinstance(data, dict):
        raise ValueError("JSON phải là object.")
    if "charges" not in data or "target_point_spec" not in data:
        raise ValueError("Thiếu trường 'charges' hoặc 'target_point_spec'.")

    charges = data["charges"]
    if not isinstance(charges, list):
        raise ValueError("'charges' phải là mảng.")
    for i, q in enumerate(charges):
        if not all(k in q for k in ("id", "value", "position")):
            raise ValueError(f"charge {i} thiếu trường bắt buộc.")
        if not isinstance(q["id"], str):
            raise ValueError(f"id của charge {i} phải là string.")
        if not isinstance(q["value"], (int, float)):
            raise ValueError(f"value của charge {i} phải là số.")
        pos = q["position"]
        if not isinstance(pos, (list, tuple)) or len(pos) != 2:
            raise ValueError(f"position của charge {i} phải là [x, y].")
        if any(not isinstance(coord, (int, float)) for coord in pos):
            raise ValueError(f"Tọa độ của charge {i} phải là số.")

    spec = data["target_point_spec"]
    if not isinstance(spec, dict):
        raise ValueError("'target_point_spec' phải là object.")
    if "type" not in spec:
        raise ValueError("'target_point_spec' thiếu 'type'.")
    if spec["type"] not in ("explicit", "equations"):
        raise ValueError("type phải là 'explicit' hoặc 'equations'.")

    if spec["type"] == "explicit":
        if "coordinates" not in spec:
            raise ValueError("type explicit cần 'coordinates'.")
        coords = spec["coordinates"]
        if not isinstance(coords, (list, tuple)) or len(coords) != 2:
            raise ValueError("coordinates phải là [x, y].")
        if any(not isinstance(c, (int, float)) for c in coords):
            raise ValueError("Tọa độ trong coordinates phải là số.")
    else:  # equations
        if "equations" not in spec:
            raise ValueError("type equations cần 'equations'.")
        eqs = spec["equations"]
        if not isinstance(eqs, list) or not all(isinstance(e, str) for e in eqs):
            raise ValueError("equations phải là list of strings.")

    # Kiểm tra target_charge nếu có
    if "target_charge_id" in spec:
        if not isinstance(spec["target_charge_id"], str):
            raise ValueError("target_charge_id phải là string.")
        if "target_charge_value" not in spec:
            raise ValueError("Có target_charge_id thì cần target_charge_value.")
        if not isinstance(spec["target_charge_value"], (int, float)):
            raise ValueError("target_charge_value phải là số.")
    return True


def _parse_equation_to_sympy(eq_str):
    """
    Biến một chuỗi như "x^2 + y^2 = 0.0144" thành biểu thức sympy: x**2 + y**2 - 0.0144
    """
    # Tách hai vế bởi dấu '='
    if "=" not in eq_str:
        raise ValueError(f"Phương trình '{eq_str}' không chứa dấu '='.")
    left, right = eq_str.split("=", 1)
    left = left.strip()
    right = right.strip()

    # Thay thế ^ bằng ** cho sympy
    left = left.replace("^", "**")
    right = right.replace("^", "**")

    # Tạo biểu thức hiệu
    expr_str = f"({left}) - ({right})"
    # Dùng sympy để parse an toàn
    x, y = sp.symbols('x y')
    try:
        expr = sp.sympify(expr_str)  # sympify tự hiểu x,y
    except Exception as e:
        raise ValueError(f"Không thể parse phương trình: {eq_str}. Lỗi: {e}")
    return expr


def solve_target_position(equations):
    """
    Giải hệ phương trình (list các string) để tìm toạ độ (x, y).
    Trả về (x_val, y_val) là số thực (float). Mặc định chọn nghiệm có y >= 0 nếu cùng
    khoảng cách, hoặc nghiệm đầu tiên nếu không có điều kiện.
    """
    if len(equations) != 2:
        raise ValueError("Cần đúng 2 phương trình để xác định toạ độ (x,y).")
    x, y = sp.symbols('x y')
    eqs = [_parse_equation_to_sympy(eq) for eq in equations]
    solutions = sp.solve(eqs, (x, y), dict=True)
    if not solutions:
        raise ValueError("Hệ phương trình vô nghiệm.")
    # Lọc nghiệm thực
    real_sols = []
    for sol in solutions:
        xv = complex(sol[x]).real
        yv = complex(sol[y]).real
        if abs(complex(sol[x]).imag) < 1e-9 and abs(complex(sol[y]).imag) < 1e-9:
            real_sols.append((float(xv), float(yv)))
    if not real_sols:
        raise ValueError("Không có nghiệm thực.")
    # Ưu tiên nghiệm có y >= 0
    pos_y = [s for s in real_sols if s[1] >= 0]
    return pos_y[0] if pos_y else real_sols[0]


def extract_and_compute(parsed_json):
    """
    Nhận JSON đã parse (dict), kiểm tra, xác định toạ độ điểm cần tính,
    và gọi compute_field_or_force để trả về vector.
    """
    validate_parsed_json(parsed_json)

    charges = parsed_json["charges"]
    spec = parsed_json["target_point_spec"]

    # 1. Xác định vị trí target
    if spec["type"] == "explicit":
        target_pos = list(spec["coordinates"])
    else:  # equations
        print(spec["equations"])
        target_pos = list(solve_target_position(spec["equations"]))
        print(target_pos)
        
    # 2. Mode và target charge
    mode = "E"
    target_charge_id = None
    target_charge_value = None
    if "target_charge_id" in spec:
        mode = "F"
        target_charge_id = spec["target_charge_id"]
        target_charge_value = spec["target_charge_value"]

    # 3. Chuẩn bị charges đầy đủ cho hàm compute_field_or_force
    #    Hàm này yêu cầu target_charge có trong danh sách charges.
    #    Ta tạo bản sao và thêm vào nếu cần.
    charges_full = list(charges)  # shallow copy các dict
    if mode == "F":
        # Thêm target charge vào danh sách nếu chưa có
        if not any(q["id"] == target_charge_id for q in charges_full):
            charges_full.append({
                "id": target_charge_id,
                "value": target_charge_value,
                "position": target_pos
            })
    # 4. Gọi hàm tính toán
    if mode == "E":
        result = compute_field_or_force(charges_full, target_pos=target_pos, mode="E")
    else:  # mode == "F"
        result = compute_field_or_force(charges_full, target_charge=target_charge_id, mode="F")

    return result


# Hàm gốc (giữ nguyên)
def compute_field_or_force(charges, target_pos=None, target_charge=None, mode="E"):
    """
    mode = "E": tính điện trường tại target_pos
    mode = "F": tính lực lên target_charge
    """
    E = np.array([0.0, 0.0])

    if mode == "E":
        point = np.array(target_pos)
    elif mode == "F":
        target = None
        for q in charges:
            if q["id"] == target_charge:
                target = q
                break
        if target is None:
            raise ValueError("Không tìm thấy target_charge")
        point = np.array(target["position"])
    else:
        raise ValueError("mode phải là 'E' hoặc 'F'")

    for q in charges:
        r_vec = point - np.array(q["position"])
        r = np.linalg.norm(r_vec)
        if r == 0:
            continue
        E += k * q["value"] * r_vec / (r**3)

    if mode == "E":
        return E
    else:  # mode = F
        q_target = None
        for q in charges:
            if q["id"] == target_charge:
                q_target = q["value"]
                break
        return q_target * E
    
import re
import json

import re
import math
import json

def sanitize_json_with_math(json_str):
    """
    - Tìm tất cả các biểu thức số học đơn giản (chứa sqrt, *, /, +, -, số) trong chuỗi JSON.
    - Thay thế chúng bằng kết quả đã tính toán một cách an toàn.
    - Trả về chuỗi JSON chỉ chứa số thuần.
    """

    # Định nghĩa các hàm toán được phép
    safe_functions = {
        "sqrt": math.sqrt,
        "sin": math.sin,
        "cos": math.cos,
        "tan": math.tan,
        "pi": math.pi,
        "e": math.e
    }

    # Hàm eval an toàn
    def safe_eval(expr_str):
        # Chỉ cho phép ký tự số, toán tử, khoảng trắng, dấu chấm, sqrt, và các hàm trên
        if not re.match(r'^[\d\.\s\+\-\*/\(\)a-z_]+$', expr_str):
            return expr_str  # có ký tự lạ thì giữ nguyên
        try:
            # eval với __builtins__ rỗng, chỉ truyền safe_functions
            result = eval(expr_str, {"__builtins__": {}}, safe_functions)
            return str(result)
        except Exception:
            return expr_str  # nếu lỗi thì giữ nguyên

    # Bước 1: Thay thế từng lời gọi sqrt( số )
    #   sqrt(3) -> số
    json_str = re.sub(
        r'sqrt\((\d+(?:\.\d+)?)\)',
        lambda m: str(math.sqrt(float(m.group(1)))),
        json_str
    )

    # Bước 2: Tìm các cụm dạng số, dấu chấm, và sqrt đã được thay bằng số
    #   Ví dụ: 0.03 * 1.7320508075688772  (sau bước 1)
    #   Hoặc vẫn còn sqrt nếu bên trong có biểu thức? Nhưng giả định sqrt luôn là sqrt(số)
    #   Ta tìm tất cả các chuỗi liên tục gồm số, dấu chấm, +, -, *, /, khoảng trắng
    #   và thay thế bằng safe_eval.
    def replace_math_expr(m):
        expr = m.group(0)
        # Bỏ khoảng trắng thừa để kiểm tra xem có vẻ là biểu thức không
        stripped = re.sub(r'\s+', '', expr)
        if not re.search(r'[+\-*/]', stripped):
            return expr  # chỉ là số đơn, không cần eval
        return safe_eval(stripped)

    # Pattern: bắt đầu bởi một số, theo sau có thể là các toán tử và số
    # Hơi rộng nhưng an toàn vì safe_eval sẽ từ chối nếu có ký tự lạ
    json_str = re.sub(
        r'(?<=[\s:,\[])(\d+(?:\.\d+)?(?:\s*[+\-*/]\s*\d+(?:\.\d+)?)+(?:\s*[+\-*/]\s*\d+(?:\.\d+)?)*)',
        replace_math_expr,
        json_str
    )

    return json_str
def extract_json_from_llm_response(response_text):
    match = re.search(r'output:\s*\n?(.*)', response_text, re.DOTALL)
    if not match:
        raise ValueError("Không tìm thấy 'output:'.")
    json_part = match.group(1).strip()

    # Loại bỏ markdown code fences
    json_part = re.sub(r'```(?:json)?\s*|\s*```', '', json_part)

    # 👇 Sửa biểu thức toán học trước khi parse
    json_part = sanitize_json_with_math(json_part)

    try:
        return json.loads(json_part)
    except json.JSONDecodeError as e:
        raise ValueError(f"JSON sau khi xử lý vẫn lỗi: {e}")

res = extract_and_compute(extract_json_from_llm_response(problem_extracter(problem_text)))
print(res)

['x^2 + y^2 = 0.08^2', '(x - 0.16)^2 + y^2 = 0.08^2']
[0.08, 0.0]
[0. 0.]
